In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
def normalize_series(s):
    """Lowercase and strip while preserving NaN values."""
    return s.where(s.isna(), s.astype(str).str.lower().str.strip())

In [4]:

def clean_ebay_to_auto24_format(input_csv_path, output_csv_path=None):
    print(f"Reading eBay data from: {input_csv_path}")
    df = pd.read_csv(input_csv_path, encoding="latin-1", low_memory=False)
    out = pd.DataFrame()
    # 1. Make & Model (normalized lowercase, spaces instead of underscores)
    out["make"] = (
        normalize_series(df["brand"])
        .str.replace("_", " ", regex=False)
        .replace({"sonstige autos": "other"})
    )
    out["model"] = normalize_series(df["model_final"])
    # 2. Numeric features & target
    out["year"] = pd.to_numeric(df["yearOfRegistration"], errors="coerce")
    out["month"] = pd.to_numeric(df["monthOfRegistration"], errors="coerce")
    out["price_eur"] = pd.to_numeric(df["price"], errors="coerce")
    out["log_price"] = np.log1p(out["price_eur"])
    out["mileage_km"] = pd.to_numeric(df["kilometer"], errors="coerce")
    out["power_ps"] = pd.to_numeric(df["powerPS"], errors="coerce")
    out["power_kw"] = (out["power_ps"] / 1.35962).round(1)
    # 3. Fuel type (German -> English)
    fuel_map = {
        "benzin": "petrol",
        "diesel": "diesel",
        "hybrid": "hybrid",
        "elektro": "electric",
        "lpg": "lpg",
        "cng": "cng",
        "andere": "other",
    }
    out["fuel_type"] = normalize_series(df["fuelType"]).map(fuel_map)
    # 4. Transmission (German -> English)
    transmission_map = {
        "manuell": "manual",
        "automatik": "automatic",
    }
    out["transmission"] = normalize_series(df["gearbox"]).map(transmission_map)
    # 5. Body type (German -> English)
    body_map = {
        "limousine": "sedan",
        "kombi": "wagon",
        "kleinwagen": "hatchback",
        "bus": "minivan",
        "suv": "suv",
        "coupe": "coupe",
        "cabrio": "cabrio",
        "andere": "other",
    }
    out["body_type"] = normalize_series(df["vehicleType"]).map(body_map)
    # 6. Damaged status (ja/nein -> True/False, preserving NaN)
    damage_map = {"ja": True, "nein": False}
    out["damaged"] = normalize_series(df["notRepairedDamage"]).map(damage_map)
    # 7. Drop invalid prices if any exist
    out = out.dropna(subset=["price_eur"]).copy()
    out = out[out["price_eur"] > 100].copy()
    if output_csv_path:
        out.to_csv(output_csv_path, index=False)
        print(f"Saved English/Auto24-formatted eBay CSV to: {output_csv_path} (Rows: {len(out)})")
    return out



In [3]:
# Example Usage:
ebay_in = r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\ebay\models_cleaned_over_10_instances.csv"
ebay_out = r"c:\Users\Andi\learn_projects\GIT\UsedCarPrice\ebay\ebay_cleaned_english_auto24_format.csv"
df_ebay_english = clean_ebay_to_auto24_format(ebay_in, ebay_out)
print(df_ebay_english.head())

NameError: name 'clean_ebay_to_auto24_format' is not defined